# Assignment 2, Problem 2: Variable Elimination

In this problem you will implement the variable elimination algorithm for exact inference in a Bayesian network.

The notebook has four parts:
1. **Classes (do not edit).** `Variable`, `Factor` and `BN`.
2. **Functions to implement.** `normalize`, `restrict_factor`, `sum_out_variable`, `multiply_factors` and `VE`.
3. **Sample network (do not edit).** A small network you can use for testing.
4. **Tests.** Run these to check your code.

Use only the Python standard library. Do not modify any input factor in your functions; always create and return new ones.

## Part 1: Classes (do not edit)

In [1]:
import math
from itertools import product


class Variable:
    """A discrete random variable with a name and a finite domain."""

    def __init__(self, name, domain):
        self.name = name
        self._domain = list(domain)
        self._evidence = None          # None means no evidence has been set

    def domain(self):
        return list(self._domain)

    def set_evidence(self, value):
        assert value in self._domain, f"{value} is not in the domain of {self.name}"
        self._evidence = value

    def clear_evidence(self):
        self._evidence = None

    def get_evidence(self):
        return self._evidence

    def __repr__(self):
        return f"Variable({self.name})"


class Factor:
    """
    A factor over an ordered list of variables (its scope).
    The values are stored in a dictionary that maps a tuple of values
    (one per variable in the scope, in scope order) to a number.
    A factor with an empty scope has the single key ().
    """

    def __init__(self, name, scope):
        self.name = name
        self.scope = list(scope)
        self.values = {}

    def get_scope(self):
        return list(self.scope)

    def set_value(self, assignment, value):
        self.values[tuple(assignment)] = value

    def get_value(self, assignment):
        return self.values[tuple(assignment)]

    def print_table(self):
        names = [v.name for v in self.scope]
        print(f"Factor {self.name}  scope = {names}")
        for assignment, value in self.values.items():
            print("  ", assignment, "->", round(value, 6))


class BN:
    """A Bayesian network: a list of variables and a list of factors (the CPTs)."""

    def __init__(self, name, variables, factors):
        self.name = name
        self._variables = list(variables)
        self._factors = list(factors)

    def variables(self):
        return list(self._variables)

    def factors(self):
        return list(self._factors)

    def clear_evidence(self):
        for v in self._variables:
            v.clear_evidence()

## Part 2: Functions to implement

Fill in the five functions below. A short description is given in each docstring.

In [2]:
def normalize(nums):
    """Return a new list proportional to nums that sums to one."""
    total = sum(nums)
    if total == 0:
        raise ValueError("Cannot normalize numbers whose sum is zero.")
    probabilities = []
    for number in nums:
        probabilities.append(number / total)
    return probabilities


def restrict_factor(f, var, value):
    """Return a new factor with var fixed to value and removed from its scope."""
    position = f.scope.index(var)
    if value not in var.domain():
        raise ValueError("Evidence value is outside the variable's domain.")
    new_scope = f.get_scope()
    new_scope.remove(var)
    result = Factor("restricted", new_scope)
    for assignment, probability in f.values.items():
        if assignment[position] == value:
            new_assignment = assignment[:position] + assignment[position + 1:]
            result.set_value(new_assignment, probability)
    return result


def sum_out_variable(f, var):
    """Return a new factor with var removed by summing over its values."""
    position = f.scope.index(var)
    new_scope = f.get_scope()
    new_scope.remove(var)
    result = Factor("summed", new_scope)
    for assignment, probability in f.values.items():
        new_assignment = assignment[:position] + assignment[position + 1:]
        previous_total = result.values.get(new_assignment, 0)
        result.set_value(new_assignment, previous_total + probability)
    return result


def multiply_factors(factors):
    """Return a new factor containing the product of all input factors."""
    scope = []
    for factor in factors:
        for var in factor.scope:
            if var not in scope:
                scope.append(var)
    result = Factor("product", scope)
    domains = []
    for var in scope:
        domains.append(var.domain())
    for assignment in product(*domains):
        probability = 1.0
        for factor in factors:
            factor_assignment = []
            for var in factor.scope:
                position = scope.index(var)
                factor_assignment.append(assignment[position])
            probability *= factor.get_value(factor_assignment)
        result.set_value(assignment, probability)
    return result


def VE(bayes_net, var_query, varlist_evidence):
    """Return the query distribution in domain order without changing input factors."""
    if var_query not in bayes_net.variables():
        raise ValueError("The query variable must belong to the network.")
    factors = bayes_net.factors()
    for evidence in varlist_evidence:
        if evidence not in bayes_net.variables():
            raise ValueError("Evidence variables must belong to the network.")
        value = evidence.get_evidence()
        if value not in evidence.domain():
            raise ValueError("Set an evidence value before calling VE.")
        restricted_factors = []
        for factor in factors:
            if evidence in factor.scope:
                factor = restrict_factor(factor, evidence, value)
            restricted_factors.append(factor)
        factors = restricted_factors
    for var in bayes_net.variables():
        if var == var_query or var in varlist_evidence:
            continue
        relevant_factors = []
        remaining_factors = []
        for factor in factors:
            if var in factor.scope:
                relevant_factors.append(factor)
            else:
                remaining_factors.append(factor)
        if relevant_factors:
            combined = multiply_factors(relevant_factors)
            reduced = sum_out_variable(combined, var)
            remaining_factors.append(reduced)
        factors = remaining_factors
    final_factor = multiply_factors(factors)
    weights = []
    for value in var_query.domain():
        if var_query in varlist_evidence:
            weight = 0.0
            if value == var_query.get_evidence():
                weight = final_factor.get_value(())
        else:
            weight = final_factor.get_value((value,))
        weights.append(weight)
    return normalize(weights)


## Part 3: Sample network (do not edit)

A five-variable network used as a standard textbook example. Burglary (`B`) and Earthquake (`E`) can set off the Alarm (`A`), and the alarm can make John (`J`) and Mary (`M`) call. All variables are Boolean.

The helper `make_cpt` builds a factor from a table giving $P(\text{child} = \text{True} \mid \text{parents})$.

In [3]:
def make_cpt(name, child, parents, p_true):
    """
    p_true maps a tuple of parent values to P(child = True | parents).
    For a variable with no parents use the key ().
    The factor scope is [child] + parents.
    """
    f = Factor(name, [child] + parents)
    for parent_vals, p in p_true.items():
        f.set_value((True,) + parent_vals, p)
        f.set_value((False,) + parent_vals, 1 - p)
    return f


B = Variable("Burglary", [True, False])
E = Variable("Earthquake", [True, False])
A = Variable("Alarm", [True, False])
J = Variable("JohnCalls", [True, False])
M = Variable("MaryCalls", [True, False])

F_B = make_cpt("P(B)", B, [], {(): 0.001})
F_E = make_cpt("P(E)", E, [], {(): 0.002})
F_A = make_cpt("P(A|B,E)", A, [B, E], {
    (True, True): 0.95, (True, False): 0.94,
    (False, True): 0.29, (False, False): 0.001})
F_J = make_cpt("P(J|A)", J, [A], {(True,): 0.90, (False,): 0.05})
F_M = make_cpt("P(M|A)", M, [A], {(True,): 0.70, (False,): 0.01})

alarm_net = BN("Alarm", [B, E, A, J, M], [F_B, F_E, F_A, F_J, F_M])

## Part 4: Tests

These tests check each function on small examples and then check `VE` on several queries of the sample network. A passing test is a good sign, not a proof, so add your own tests.

In [4]:
def close(a, b, tol=1e-9):
    return math.isclose(a, b, rel_tol=tol, abs_tol=tol)


# two small variables and two factors for testing the factor operations
X = Variable("X", [0, 1])
Y = Variable("Y", [0, 1])

f = Factor("f", [X, Y])
for (x, y), val in {(0, 0): 0.1, (0, 1): 0.9, (1, 0): 0.2, (1, 1): 0.8}.items():
    f.set_value((x, y), val)

g = Factor("g", [Y])
g.set_value((0,), 0.3)
g.set_value((1,), 0.7)

# normalize
out = normalize([1, 3])
assert close(out[0], 0.25) and close(out[1], 0.75), "normalize failed"

# restrict_factor: f restricted to Y = 1 should have scope [X] with values 0.9, 0.8
r = restrict_factor(f, Y, 1)
assert r.scope == [X], "restrict_factor: wrong scope"
assert close(r.get_value((0,)), 0.9) and close(r.get_value((1,)), 0.8), "restrict_factor: wrong values"
assert len(f.values) == 4 and f.scope == [X, Y], "restrict_factor modified its input"

# sum_out_variable: summing Y out of f gives 1.0 and 1.0; summing X out gives 0.3 and 1.7
s = sum_out_variable(f, Y)
assert s.scope == [X], "sum_out_variable: wrong scope"
assert close(s.get_value((0,)), 1.0) and close(s.get_value((1,)), 1.0), "sum_out_variable: wrong values"
s2 = sum_out_variable(f, X)
assert close(s2.get_value((0,)), 0.3) and close(s2.get_value((1,)), 1.7), "sum_out_variable: wrong values"
assert len(f.values) == 4 and f.scope == [X, Y], "sum_out_variable modified its input"

# multiply_factors: scope is the union of the scopes, values are products
p = multiply_factors([f, g])
assert set(p.scope) == {X, Y}, "multiply_factors: wrong scope"
lookup = lambda x, y: p.get_value(tuple({X: x, Y: y}[v] for v in p.scope))
assert close(lookup(0, 0), 0.03) and close(lookup(0, 1), 0.63), "multiply_factors: wrong values"
assert close(lookup(1, 0), 0.06) and close(lookup(1, 1), 0.56), "multiply_factors: wrong values"
assert len(f.values) == 4 and len(g.values) == 2, "multiply_factors modified its input"

print("Factor operation tests passed.")

Factor operation tests passed.


In [5]:
def run_query(bn, query, evidence):
    """evidence is a dict {variable: value}. Returns the result of VE."""
    bn.clear_evidence()
    for var, val in evidence.items():
        var.set_evidence(val)
    result = VE(bn, query, list(evidence.keys()))
    bn.clear_evidence()
    return result


# (query variable, evidence, expected probabilities in domain order [True, False])
cases = [
    (B, {}, [0.001, 0.999]),
    (J, {}, [0.05214, 0.94786]),
    (A, {J: True}, [0.04344, 0.95656]),
    (E, {A: True}, [0.23101, 0.76899]),
    (E, {A: True, B: True}, [0.00202, 0.99798]),
    (B, {J: True, M: False}, [0.00513, 0.99487]),
    (M, {B: True, E: False}, [0.6586, 0.3414]),
]

all_ok = True
for query, evidence, expected in cases:
    got = run_query(alarm_net, query, evidence)
    ok = len(got) == len(expected) and all(close(a, b, 1e-4) for a, b in zip(got, expected))
    all_ok = all_ok and ok
    label = ", ".join(f"{v.name}={val}" for v, val in evidence.items()) or "no evidence"
    print(("PASS" if ok else "FAIL"), f"P({query.name} | {label}) = {[round(x, 5) for x in got]}")
assert all_ok, "At least one VE query gave the wrong answer."

# VE must not change the network's factors
before = [dict(fac.values) for fac in alarm_net.factors()]
run_query(alarm_net, B, {J: True})
after = [dict(fac.values) for fac in alarm_net.factors()]
assert before == after, "VE modified a factor of the Bayes net."
print("All tests passed.")

PASS P(Burglary | no evidence) = [0.001, 0.999]
PASS P(JohnCalls | no evidence) = [0.05214, 0.94786]
PASS P(Alarm | JohnCalls=True) = [0.04344, 0.95656]
PASS P(Earthquake | Alarm=True) = [0.23101, 0.76899]
PASS P(Earthquake | Alarm=True, Burglary=True) = [0.00202, 0.99798]
PASS P(Burglary | JohnCalls=True, MaryCalls=False) = [0.00513, 0.99487]
PASS P(MaryCalls | Burglary=True, Earthquake=False) = [0.6586, 0.3414]
All tests passed.


## Additional tests
Check edge cases and compare VE against direct enumeration for every evidence pattern in the sample network.

In [6]:
# Extra checks: edge cases, independent enumeration, and unchanged inputs.
numbers = [2, 6]
assert normalize(numbers) == [0.25, 0.75] and numbers == [2, 6]
for numbers in ([], [0, 0]):
    try:
        normalize(numbers)
    except ValueError:
        pass
    else:
        raise AssertionError("A zero total must raise ValueError.")

assert multiply_factors([]).values == {(): 1.0}
assert restrict_factor(g, Y, 1).values == {(): 0.7}
assert close(sum_out_variable(g, Y).get_value(()), 1.0)

# Non-Boolean domains and opposite scope orders catch indexing mistakes.
T = Variable("T", ["red", "green", "blue"])
U = Variable("U", [0, 1])
h = Factor("h", [T, U])
k = Factor("k", [U, T])
for i, color in enumerate(T.domain()):
    for bit in U.domain():
        h.set_value((color, bit), i + bit + 1)
        k.set_value((bit, color), 2 * i + bit + 1)
snapshots = [(factor.get_scope(), dict(factor.values)) for factor in [h, k]]
combined = multiply_factors([h, k])
for color, bit in product(T.domain(), U.domain()):
    assert combined.get_value((color, bit)) == h.get_value((color, bit)) * k.get_value((bit, color))
assert restrict_factor(h, T, "green").values == {(0,): 2, (1,): 3}
assert sum_out_variable(h, T).values == {(0,): 6, (1,): 9}
assert snapshots == [(factor.get_scope(), dict(factor.values)) for factor in [h, k]]

# Compute the joint directly, independently of all five solution functions.
variables = alarm_net.variables()
original = [(factor.get_scope(), dict(factor.values)) for factor in alarm_net.factors()]
query_count = 0
for query in variables:
    for settings in product([None, True, False], repeat=len(variables)):
        alarm_net.clear_evidence()
        evidence = []
        for var, setting in zip(variables, settings):
            if setting is not None:
                var.set_evidence(setting)
                evidence.append(var)
        expected = [0.0] * len(query.domain())
        for values in product(*[var.domain() for var in variables]):
            assignment = dict(zip(variables, values))
            if any(assignment[var] != var.get_evidence() for var in evidence):
                continue
            joint = 1.0
            for factor in alarm_net.factors():
                joint *= factor.get_value([assignment[var] for var in factor.scope])
            index = query.domain().index(assignment[query])
            expected[index] += joint
        total = sum(expected)
        expected = [weight / total for weight in expected]
        got = VE(alarm_net, query, evidence)
        assert all(close(actual, target) for actual, target in zip(got, expected))
        assert all(var.get_evidence() == setting for var, setting in zip(variables, settings))
        query_count += 1
alarm_net.clear_evidence()
assert original == [(factor.get_scope(), dict(factor.values)) for factor in alarm_net.factors()]

# Impossible evidence must still fail when its factor is disconnected from the query.
Z = Variable("Z", [True, False])
zero_evidence = make_cpt("P(Z)", Z, [], {(): 0.0})
Z.set_evidence(True)
impossible_net = BN("Impossible", [B, Z], [F_B, zero_evidence])
for query in [B, Z]:
    try:
        VE(impossible_net, query, [Z])
    except ValueError:
        pass
    else:
        raise AssertionError("Impossible evidence must raise ValueError.")
Z.clear_evidence()
try:
    VE(impossible_net, B, [Z])
except ValueError:
    pass
else:
    raise AssertionError("Unset evidence must raise ValueError.")
print(f"Extra tests passed, including {query_count} queries checked by direct enumeration.")


Extra tests passed, including 1215 queries checked by direct enumeration.
